## Context

These are bench notes from a Wednesday afternoon that I didn't expect to turn into anything. I was running a standard GUV encapsulation of our PURE system mix, this time with a Cy3-labeled poly-A(50) oligo added as a passive volume tracer. The idea was simply to confirm encapsulation efficiency before committing reagents to a full eGFP expression run.

About 40 minutes into imaging, I noticed bright puncta forming inside a subset of vesicles — not aggregates stuck to the membrane, but interior foci that were roundish, mobile by Brownian drift, and clearly distinct from the diffuse Cy3 background. My first instinct was contamination or RNA degradation products. But the puncta appeared in ~30% of vesicles above a certain size threshold, and they weren't present in the control wells without poly-A.

A quick literature check suggested this might be crowding-induced liquid-liquid phase separation of the poly-A RNA, possibly nucleated by the PEG-8000 we include in our encapsulation buffer. RNA condensates driven by crowding agents have been reported in bulk (Maharana et al., 2018; Boija et al., 2018), but I haven't seen it documented specifically in the GUV confinement context. Writing this up as a lab note in case it's relevant to the Cambridge RNA condensate project.


## Methods

No formal protocol deviation — this was the standard DOPC:DOPG (9:1) GUV preparation by PVA gel-assisted swelling that we run every week. The only difference was the addition of Cy3-labeled poly-A(50) RNA (IDT, HPLC-purified) to the PURE system mix at concentrations of 0.1, 0.5, and 1.0 mg/mL.

Imaging was on the Zeiss LSM 980 confocal (40× water objective, NA 1.2). I used a single-channel acquisition at 561 nm excitation / 580–620 nm emission. No deconvolution was applied. Images were taken at 5-minute intervals starting immediately after GUVs were transferred to the observation chamber.

For the condensate intensity measurements in the figure, I manually selected 12 vesicles that showed clear puncta (>3 foci per vesicle) and tracked mean puncta intensity vs. diffuse cytoplasmic signal over time using a custom FIJI macro. The ratio (puncta intensity / background) is what's plotted — it's a rough proxy for condensate enrichment rather than a calibrated concentration measurement.

I haven't done FRAP yet to confirm liquid-like behavior. That's on the list.


## Results

The 0.1 mg/mL poly-A wells looked clean — uniform Cy3 fluorescence throughout the vesicle lumen, no puncta at any timepoint. The 0.5 and 1.0 mg/mL wells were where things got interesting.

In the 0.5 mg/mL condition, foci appeared in roughly 28% of GUVs (estimated from a single field of view, n ≈ 60 vesicles). They nucleated gradually — you could watch the diffuse background dim as material condensed into 2–5 bright spots per vesicle over 20–30 minutes. In the 1.0 mg/mL condition, puncta were more numerous (often >8 per vesicle) and appeared faster, within the first 10 minutes of imaging.

The enrichment ratio in tracked vesicles reached ~4.5× at plateau, meaning condensate puncta were ~4–5× brighter than the surrounding cytoplasm on a per-pixel basis. Puncta were dynamic — they moved around the interior and occasionally fused into larger droplets, which is the classic hallmark of liquid-liquid phase separation rather than irreversible aggregation.

One odd thing: the condensates seemed to preferentially form near the inner membrane leaflet in the early stages before migrating to the interior. Not sure what to make of that yet — could be a membrane-RNA interaction, or it could just be imaging artifact from evanescent field effects near the coverslip.


## Specification

| Parameter | Value |
|---|---|
| Lipid composition | DOPC:DOPG 9:1 mol/mol |
| Swelling buffer | 300 mOsm sucrose, 10 mM HEPES pH 7.4, 4% PEG-8000 |
| RNA species | Poly-A(50), Cy3-labeled, IDT, HPLC-purified |
| RNA concentrations tested | 0.1, 0.5, 1.0 mg/mL |
| Objective | 40× water, NA 1.2 |
| Excitation | 561 nm |
| Emission | 580–620 nm |
| Imaging interval | 5 min |
| Total imaging duration | 90 min |
| Temperature | 25 °C (ambient, uncontrolled) |
| Condensate foci threshold | >3 foci per vesicle |
| Vesicles tracked for kinetics | 12 |
| Peak enrichment ratio | ~4.5× (puncta / background) |
| FRAP performed | No (pending) |

> **Note:** Temperature was not actively controlled in this experiment. Lab ambient was approximately 22–24 °C. All follow-up experiments should use the stage incubator.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

os.makedirs('figures', exist_ok=True)

rng = np.random.default_rng(99)
time = np.linspace(0, 90, 19)  # 0 to 90 min, 5 min intervals

# Condensate enrichment ratio over time for 0.5 and 1.0 mg/mL poly-A
def condensate_kinetics(t, onset, rate, plateau):
    t_adj = np.maximum(t - onset, 0)
    return 1.0 + (plateau - 1.0) * (1 - np.exp(-rate * t_adj))

trace_05 = condensate_kinetics(time, onset=18, rate=0.10, plateau=3.8)
trace_10 = condensate_kinetics(time, onset=8, rate=0.14, plateau=4.5)
noise_05 = rng.normal(0, 0.15, len(time))
noise_10 = rng.normal(0, 0.18, len(time))

mpl.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(time, trace_05 + noise_05, color='#A78BFA', linewidth=2, marker='o',
        markersize=4, label='0.5 mg/mL poly-A')
ax.plot(time, trace_10 + noise_10, color='#7C3AED', linewidth=2, marker='s',
        markersize=4, label='1.0 mg/mL poly-A')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1, label='No enrichment (ratio = 1)')
ax.set_xlabel('Time (min)', labelpad=8)
ax.set_ylabel('Condensate Enrichment Ratio\n(puncta intensity / cytoplasmic background)', labelpad=8)
ax.set_title('RNA Condensate Formation in GUV-Confined Poly-A RNA', fontsize=11, pad=10)
ax.set_xlim(0, 90)
ax.set_ylim(0.5, 5.5)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig('figures/key-result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/key-result.png')
